# Group Lab 5: Geostatistics, Semivariograms, IDW, and Spatial Prediction

        **Week:** Week 11

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Calculate distances.
- Build an empirical semivariogram.
- Predict with IDW.
- Validate with leave-one-out cross-validation.

        ## Earth and environmental motivation

        Spatial prediction helps estimate environmental variables between observations, but the result depends on distance assumptions and validation.

        ## Dataset

        `data/processed/synthetic_soil_moisture_spatial.csv`

        ## Python concepts used

        - Distance
- Semivariogram
- IDW
- Cross-validation
- Spatial prediction

## Group Lab 4 Debrief and Collaborative Debugging (First 10 Minutes)

Open the debrief card from Group Lab 4. Two to four students or groups will share a solved problem, an unresolved problem with evidence, or a verification choice. Work one unresolved problem together, then report to the class.

- 0-5 min: student discussion. Compare cards in small groups and
  debug one unresolved problem together.
- 5-10 min: student reports. Two to four groups report, and the
  class records one reusable lesson.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Group roles

- Module A: spatial data preparation and quality control.
- Module B: semivariogram and IDW implementation.
- Module C: prediction map, validation, and interpretation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from earthcourse.spatial import empirical_semivariogram, idw_predict, leave_one_out_idw
from earthcourse.stats import rmse

points = pd.read_csv(PROCESSED_DIR / "synthetic_soil_moisture_spatial.csv")
print(points.head())

In [ ]:
variogram = empirical_semivariogram(points["x_km"], points["y_km"], points["soil_moisture"], n_bins=10)
print(variogram)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(variogram["distance_mid"], variogram["semivariance"], marker="o")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Semivariance")
ax.set_title("Empirical semivariogram for synthetic soil moisture")
fig.tight_layout()
plt.show()

In [ ]:
gx = np.linspace(points["x_km"].min(), points["x_km"].max(), 40)
gy = np.linspace(points["y_km"].min(), points["y_km"].max(), 40)
xx, yy = np.meshgrid(gx, gy)
pred = idw_predict(points["x_km"], points["y_km"], points["soil_moisture"], xx.ravel(), yy.ravel()).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.contourf(xx, yy, pred, levels=15, cmap="YlGnBu")
ax.scatter(points["x_km"], points["y_km"], c=points["soil_moisture"], edgecolor="black", cmap="YlGnBu", s=35)
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("IDW-predicted soil moisture")
fig.colorbar(image, ax=ax, label="Soil moisture")
fig.tight_layout()
plt.show()

In [ ]:
loo = leave_one_out_idw(points["x_km"], points["y_km"], points["soil_moisture"])
print(loo.head())
print("LOO RMSE:", rmse(loo["observed"], loo["predicted"]))

## Guided coding: how the power parameter changes the map (Module B)

The IDW power controls how quickly influence fades with distance. Low power
borrows from far away and smooths the map; high power trusts only the
nearest points and draws bullseyes around them. Leave-one-out RMSE says
which choice predicts best here.

In [ ]:
powers = [1, 2, 4]
loo_rows = []
vmin = points["soil_moisture"].min()
vmax = points["soil_moisture"].max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, power in zip(axes, powers):
    pred_power = idw_predict(points["x_km"], points["y_km"], points["soil_moisture"],
                             xx.ravel(), yy.ravel(), power=power).reshape(xx.shape)
    image = ax.contourf(xx, yy, pred_power, levels=15, cmap="YlGnBu", vmin=vmin, vmax=vmax)
    ax.scatter(points["x_km"], points["y_km"], c="black", s=6)
    ax.set_title(f"IDW power = {power}")
    ax.set_xlabel("x (km)")
    loo_power = leave_one_out_idw(points["x_km"], points["y_km"], points["soil_moisture"], power=power)
    loo_rows.append({"power": power,
                     "loo_rmse": round(rmse(loo_power["observed"], loo_power["predicted"]), 4)})
axes[0].set_ylabel("y (km)")
fig.colorbar(image, ax=list(axes), label="Soil moisture", shrink=0.9)
plt.show()

print(pd.DataFrame(loo_rows))

## Guided coding: validation scatter with a 1:1 line (Module C)

Points below the 1:1 line are under-predicted, points above are
over-predicted. IDW pulls predictions toward the local mean, so expect high
observations under-predicted and low ones over-predicted; that compression
is the signature of a smoothing interpolator.

In [ ]:
from earthcourse.stats import mae

loo_p2 = leave_one_out_idw(points["x_km"], points["y_km"], points["soil_moisture"], power=2)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(loo_p2["observed"], loo_p2["predicted"], s=18, alpha=0.7)
limits = [points["soil_moisture"].min(), points["soil_moisture"].max()]
ax.plot(limits, limits, color="firebrick", linewidth=1, label="1:1 line")
ax.set_xlabel("Observed soil moisture")
ax.set_ylabel("LOO-predicted soil moisture")
ax.set_title("Leave-one-out validation, IDW power 2")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
plt.show()
print(f"LOO RMSE: {rmse(loo_p2['observed'], loo_p2['predicted']):.4f}")
print(f"LOO MAE:  {mae(loo_p2['observed'], loo_p2['predicted']):.4f}")

## Reading the semivariogram: nugget, sill, and range

Three features carry the spatial story. The **nugget** is the semivariance
near zero distance (measurement noise plus variation finer than the sampling).
The **sill** is the plateau (the overall variance of the field). The
**range** is the distance where the curve reaches the sill; beyond it, two
points are effectively unrelated, and interpolation between them has little
support. Estimate all three from your plot before trusting any map.

In [ ]:
print(variogram[["distance_mid", "semivariance", "pair_count"]])
near_sill = variogram.loc[variogram["semivariance"] >= 0.95 * variogram["semivariance"].max(),
                          "distance_mid"]
print(f"\nApproximate range (first bin at 95 percent of maximum semivariance): "
      f"{near_sill.iloc[0]:.2f} km")

## Optional responsible AI use

Use the workflow from Group Lab 4 to review one function, add one test, improve one figure, or improve documentation. Update `AI_USAGE_LOG.md` if used.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an unchanged guided notebook does not meet the submission requirement.

Run leave-one-out IDW for powers 1, 2, 3, and 4. Make an RMSE table and choose a power using the validation result. Add one spatial or scientific reason why the lowest RMSE alone may not settle the choice.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Rebuild the semivariogram with `max_distance=5.0` and with the default.
   Which bins gain or lose pairs, and which version estimates the range more
   clearly?
2. Run leave-one-out validation at power 8. IDW with a very high power
   behaves like nearest-neighbor assignment; how does its RMSE compare, and
   what does the map look like?
3. Group the points by `landcover_code` and compare mean soil moisture.
   Should land cover enter the interpolation, and what method would allow
   that?

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Empirical semivariogram plot
- [ ] IDW prediction map
- [ ] Leave-one-out validation result
- [ ] Scientific interpretation

        ## Short reflection

        Where is IDW likely to be least reliable in the map, and why?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a valid and useful report.

**Goal:** Replace this text.

**Expected result:** Replace this text.

**What happened:** Replace this text.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
